In [ ]:
import os, glob, json, sqlite3
import numpy as np
import pandas as pd

# DATA_DIR is the search campaign directory (contains campaign.db). The
# analysis runner injects it; fall back to the current directory.
DATA_DIR = ''
db = os.path.join(DATA_DIR, 'campaign.db')
if not os.path.exists(db):
    hits = glob.glob(os.path.join(DATA_DIR or '.', '**', 'campaign.db'), recursive=True)
    db = hits[0] if hits else db

conn = sqlite3.connect(db)
conn.row_factory = sqlite3.Row
camp = conn.execute(
    "SELECT config_json FROM campaign ORDER BY created_at DESC LIMIT 1").fetchone()
config = json.loads(camp['config_json']) if camp and camp['config_json'] else {}
search = config.get('search') or {}
archive_spec = (search.get('strategy_parameters') or {}).get('archive') or {}
objectives = search.get('objectives') or [{'name': 'failure_rate', 'direction': 'maximize'}]
OBJECTIVE = objectives[0]['name']
DIRECTION = objectives[0].get('direction', 'maximize')

# One row per evaluated parameter set, tagged with the batch it was proposed in.
rows = conn.execute(
    "SELECT b.idx AS batch, u.params_json, u.objectives_json, u.measures_json "
    "FROM unit u JOIN batch b ON u.batch_id = b.id "
    "ORDER BY b.idx, u.id").fetchall()
conn.close()

records = []
for r in rows:
    rec = {'batch': r['batch']}
    rec.update(json.loads(r['measures_json'] or '{}'))
    rec.update(json.loads(r['objectives_json'] or '{}'))
    rec.update(json.loads(r['params_json'] or '{}'))
    records.append(rec)
df = pd.DataFrame(records)

# Archive behavior measures + ranges: from the stored QD config when present,
# else inferred from the data (so the notebook still works for random/optuna).
if archive_spec.get('measures'):
    MEASURES = list(archive_spec['measures'].keys())
    RANGES = [(m['low'], m['high']) for m in archive_spec['measures'].values()]
    CELLS = int(archive_spec.get('cells', 512))
else:
    known = ['max_tilt', 'drift_dist', 'landing_speed', 'control_effort']
    MEASURES = [m for m in known if m in df.columns]
    RANGES = [(float(df[m].min()), float(df[m].max())) for m in MEASURES]
    CELLS = min(512, max(16, len(df)))

# Keep only measures actually present in the recorded data.
present = [(m, r) for m, r in zip(MEASURES, RANGES) if m in df.columns]
MEASURES = [m for m, _ in present]
RANGES = [r for _, r in present]
if OBJECTIVE not in df.columns:
    obj_cols = [c for c in df.columns if c not in (['batch'] + MEASURES)]
    OBJECTIVE = obj_cols[0] if obj_cols else (df.columns[0] if len(df.columns) else 'objective')

n_batches = int(df['batch'].max()) + 1 if len(df) else 0
print(f'{len(df)} unit(s) across {n_batches} batch(es) | objective={OBJECTIVE} '
      f'({DIRECTION}) | measures={MEASURES} | cells={CELLS}')
df.head()

In [ ]:
# Search outcome: how/why this campaign stopped (search mode only). Reads the
# parseable stop record persisted on the campaign row; silently skipped for batch
# campaigns or older stores without the columns.
import os as _os, glob as _glob, sqlite3 as _sqlite3
_db = _os.path.join(DATA_DIR, 'campaign.db')
if not _os.path.exists(_db):
    _hits = _glob.glob(_os.path.join(DATA_DIR or '.', '**', 'campaign.db'), recursive=True)
    _db = _hits[0] if _hits else _db
try:
    _con = _sqlite3.connect(_db)
    _row = _con.execute('SELECT stop_kind, stop_reason, batches, elapsed_s '
                        'FROM campaign ORDER BY created_at DESC LIMIT 1').fetchone()
    _con.close()
except Exception:
    _row = None
if _row and _row[0]:
    _kind, _reason, _nb, _es = _row
    print(f'🏁 Search stopped: {_reason}')
    print(f'   {_nb} batch(es), {_es:.0f}s  (criterion: {_kind})')

In [ ]:
import itertools
import matplotlib.pyplot as plt

# The QD views need pyribs (archive + stats) and shapely (cvt_archive_heatmap);
# both ship with the `qd` extra: pip install 'robovast[qd]'.
try:
    from ribs.archives import CVTArchive
    from ribs.visualize import cvt_archive_heatmap
    HAVE_RIBS = True
except Exception as e:
    HAVE_RIBS = False
    print("pyribs/shapely unavailable -- install the qd extra: 'robovast[qd]'.", e)


def build_archive(measure_names, ranges, cells, sub_df):
    """Rebuild a CVT archive from recorded units (deterministic seed=0).

    Only (measures, objective) drive the binning, so this faithfully reproduces
    coverage / QD-score / per-cell elites without the live optimizer state.
    """
    arc = CVTArchive(solution_dim=1, cells=int(cells), ranges=list(ranges), seed=0)
    sub = sub_df.dropna(subset=list(measure_names) + [OBJECTIVE])
    if len(sub):
        obj = sub[OBJECTIVE].to_numpy(float)
        if DIRECTION == 'minimize':
            obj = -obj  # pyribs maximizes; negate so "better" is always larger
        arc.add(np.zeros((len(sub), 1)), obj, sub[list(measure_names)].to_numpy(float))
    return arc


def cvt_heatmap(ax, archive, title, vmin=None, vmax=None, cbar=True):
    """Draw a 2-D CVT archive coloured by elite objective (pyribs + shapely)."""
    cvt_archive_heatmap(archive, ax=ax, vmin=vmin, vmax=vmax,
                        cbar='auto' if cbar else None)
    ax.set_title(title)

In [ ]:
# QD progress over batches: rebuild the full N-D archive incrementally and read
# its stats after each batch. "is QD improving / converging?"
batches = sorted(int(b) for b in df['batch'].dropna().unique()) if len(df) else []
if HAVE_RIBS and MEASURES and batches:
    arc = CVTArchive(solution_dim=1, cells=CELLS, ranges=RANGES, seed=0)
    hist = []
    for b in batches:
        sub = df[df['batch'] == b].dropna(subset=MEASURES + [OBJECTIVE])
        if len(sub):
            obj = sub[OBJECTIVE].to_numpy(float)
            if DIRECTION == 'minimize':
                obj = -obj
            arc.add(np.zeros((len(sub), 1)), obj, sub[MEASURES].to_numpy(float))
        s = arc.stats
        elites = arc.data(return_type='dict').get('objective', [])
        best = max(elites) if len(elites) else float('nan')
        if DIRECTION == 'minimize' and best == best:
            best = -best
        hist.append((b, s.coverage, s.qd_score, s.num_elites, best))
    H = pd.DataFrame(hist, columns=['batch', 'coverage', 'qd_score', 'num_elites', 'best'])
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    for ax, col, lbl in zip(axes.ravel(),
                            ['coverage', 'qd_score', 'num_elites', 'best'],
                            ['coverage (fraction of cells filled)', 'QD-score',
                             'number of elites', f'best {OBJECTIVE}']):
        ax.plot(H['batch'], H[col], '-o', ms=3)
        ax.set_xlabel('batch'); ax.set_ylabel(lbl); ax.grid(alpha=0.3)
    fig.suptitle(f'QD progress over batches ({CELLS}-cell CVT, {len(MEASURES)} measures)')
    plt.tight_layout(); plt.show()
    print(H.to_string(index=False))
elif not HAVE_RIBS:
    print('QD progress needs the qd extra (pyribs + shapely).')
else:
    print('No batches recorded.')

In [ ]:
# The CVT map: the discovered failure landscape. The archive has up to 4 measures,
# so it is shown as 2-D projections, one CVT heatmap per measure pair, over ALL data.
pairs = list(itertools.combinations(range(len(MEASURES)), 2))
if HAVE_RIBS and pairs:
    vmin = float(df[OBJECTIVE].min()); vmax = float(df[OBJECTIVE].max())
    ncol = min(3, len(pairs)); nrow = (len(pairs) + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.5 * ncol, 4 * nrow), squeeze=False)
    for ax in axes.ravel()[len(pairs):]:
        ax.axis('off')
    for ax, (i, j) in zip(axes.ravel(), pairs):
        arc2 = build_archive([MEASURES[i], MEASURES[j]], [RANGES[i], RANGES[j]], CELLS, df)
        cvt_heatmap(ax, arc2, f'{MEASURES[i]} x {MEASURES[j]}', vmin=vmin, vmax=vmax)
        ax.set_xlabel(MEASURES[i]); ax.set_ylabel(MEASURES[j])
    fig.suptitle(f'CVT archive map (2-D projections), coloured by {OBJECTIVE}')
    plt.tight_layout(); plt.show()
elif not HAVE_RIBS:
    print('CVT map needs the qd extra (pyribs + shapely).')
else:
    print('CVT map needs >= 2 behavior measures.')

In [ ]:
# How the archive fills over batches: the primary measure pair, rebuilt
# cumulatively at a few checkpoints (fixed seed -> identical cells, comparable).
if HAVE_RIBS and len(MEASURES) >= 2 and batches:
    import matplotlib as mpl
    i, j = 0, 1
    mi, mj, ri, rj = MEASURES[i], MEASURES[j], RANGES[i], RANGES[j]
    vmin = float(df[OBJECTIVE].min()); vmax = float(df[OBJECTIVE].max())
    if len(batches) <= 6:
        checkpoints = batches
    else:
        checkpoints = sorted(set(np.linspace(batches[0], batches[-1], 6).astype(int)))
    fig, axes = plt.subplots(1, len(checkpoints), figsize=(4 * len(checkpoints), 4),
                             squeeze=False)
    for ax, b in zip(axes[0], checkpoints):
        arc2 = build_archive([mi, mj], [ri, rj], CELLS, df[df['batch'] <= b])
        cvt_heatmap(ax, arc2, f'after batch {b}', vmin=vmin, vmax=vmax, cbar=False)
        ax.set_xlabel(mi); ax.set_ylabel(mj)
    # One shared colourbar for the row.
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin, vmax), cmap='viridis')
    fig.colorbar(sm, ax=axes[0].tolist(), label=OBJECTIVE, fraction=0.025, pad=0.01)
    fig.suptitle(f'Archive filling over batches ({mi} x {mj}), coloured by {OBJECTIVE}')
    plt.show()
elif not HAVE_RIBS:
    print('Map-filling view needs the qd extra (pyribs + shapely).')
else:
    print('Map-filling view needs >= 2 behavior measures.')

In [ ]:
# Discovery order: every evaluated point in the primary behavior plane, coloured
# by the batch it was proposed in -- the emitter's walk through the space.
if len(MEASURES) >= 2 and len(df):
    mi, mj = MEASURES[0], MEASURES[1]
    fig, ax = plt.subplots(figsize=(6.5, 5))
    sc = ax.scatter(df[mi], df[mj], c=df['batch'], cmap='viridis', s=20, edgecolor='none')
    ax.set_xlabel(mi); ax.set_ylabel(mj)
    ax.set_title('Behavior space, coloured by batch (discovery order)')
    fig.colorbar(sc, ax=ax, label='batch')
    plt.tight_layout(); plt.show()
else:
    print('Discovery scatter needs >= 2 behavior measures.')